# 01 — ERCOT Hourly Load Inspection and Cleaning

**Purpose.** Inspect and clean historical ERCOT hourly system and weather-zone demand data (2019–2025), normalize Hour Ending notation, diagnose daylight-saving transitions, and validate the combined dataset.

**Inputs.** Annual ERCOT load Excel files in `data/` (the code discovers `*.xlsx` recursively); expected source columns are `HourEnding`, the eight weather-zone columns, and `ERCOT`. Place only the intended annual load Excel files under the local `data/` directory when running, or check discovery results before proceeding.

**Outputs.** The cleaned 2019–2025 Parquet dataset and two CSV validation reports under `data/processed/`. Source raw data are not distributed in this repository.

**Reading guide.** This notebook first demonstrates an intentionally naive timestamp parse, diagnoses why it fails for `24:00` and DST conventions, then runs the corrected reusable cleaning function over all seven years. The intermediate 366 invalid-timestamp count is **not** a final data-quality failure. The 2021 zone-total discrepancy is flagged rather than silently removed.

**Execution status.** Numerical outputs shown in this notebook are saved from the original analysis; the GitHub-preparation copy has not been re-executed with the full source data. Local machine-specific and package-installation outputs have been cleared.

**Dependencies.** Install `requirements.txt` in the selected Jupyter environment. Run from the repository root or its `notebooks/` directory, and verify that the discovered Excel files match the intended ERCOT annual load inputs.


## 1. Source Discovery and 2019 Inspection

Locate ERCOT annual load workbooks, inspect the 2019 source fields, and record initial checks before modifying timestamps.

In [ ]:
from pathlib import Path

print("Notebook execution location:")
print(("notebooks/" if Path.cwd().name == "notebooks" else "./"))

In [ ]:
from pathlib import Path

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

print("Project root:")
print("<repository root>")

In [ ]:
data_folder = project_root / "data"

print("Data folder:")
print(data_folder.relative_to(project_root).as_posix())

print(
    "Data folder exists:",
    data_folder.exists()
)

excel_files = sorted(data_folder.rglob("*.xlsx"))

print(
    "\nExcel files found:",
    len(excel_files)
)

for file in excel_files:
    print(file.relative_to(project_root))

In [4]:
files_2019 = [
    file
    for file in excel_files
    if "2019" in file.name
]

print("2019 files:")

for file in files_2019:
    print(file.relative_to(project_root))

2019 files:
data/raw/ercot_load/2019/Native_Load_2019.xlsx


In [5]:
import pandas as pd

print(
    "Pandas version:",
    pd.__version__
)

Pandas version:


**Environment note:** Install project dependencies before running the notebook with `pip install -r requirements.txt`. Do not install packages mid-analysis.

In [ ]:
if len(files_2019) == 0:
    raise FileNotFoundError(
        "No 2019 Excel file was found. "
    )

if len(files_2019) > 1:
    print(
        "Warning: Multiple 2019 files were found."
    )

file_path = files_2019[0]

print("Selected file:")
print(file_path.relative_to(project_root).as_posix())

df = pd.read_excel(file_path)

print("\nFile loaded successfully.")
print("File loaded successfully.")

In [8]:
print("Data shape:")
print(df.shape)

Data shape:
(8760, 10)


In [9]:
print("First five rows:")
df.head()

First five rows:


,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2019 01:00,9783.594839,1264.194059,3164.720629,827.759428,11697.766093,3066.891146,5993.974289,1282.542956,37081.443439
1,01/01/2019 02:00,9726.062326,1270.494294,3178.950849,830.256302,11787.936174,3140.383209,6029.773970,1295.132811,37258.989933
2,01/01/2019 03:00,9654.373521,1269.504203,3200.881160,831.505553,11861.628382,3137.721342,6043.348512,1301.225133,37300.187809
3,01/01/2019 04:00,9631.735303,1243.168422,3221.963116,839.689838,11989.105118,3105.497101,6086.408464,1305.976101,37423.543464
4,01/01/2019 05:00,9698.889228,1309.572865,3240.634870,848.672153,12219.106347,3078.256356,6180.939962,1319.140512,37895.212293


In [10]:
print("Column names:")
print(df.columns.tolist())

Column names:
['HourEnding', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']


In [11]:
print("Data types:")
print(df.dtypes)

Data types:
HourEnding        str
COAST         float64
EAST          float64
FWEST         float64
NORTH         float64
NCENT         float64
SOUTH         float64
SCENT         float64
WEST          float64
ERCOT         float64
dtype: object


In [12]:
missing_values = df.isna().sum()

print("Missing values:")
print(missing_values)

Missing values:
HourEnding    0
COAST         0
EAST          0
FWEST         0
NORTH         0
NCENT         0
SOUTH         0
SCENT         0
WEST          0
ERCOT         0
dtype: int64


## 2. Initial Data Quality Assessment

Evaluate completeness, chronological order, duplicate records, timestamp parsing, plausible load values, and reconciliation between the reported system load and eight weather-zone totals. The first pass intentionally exposes the Hour Ending formatting issues before normalization.

In [14]:
# Keep an untouched copy of the loaded source data.
df_raw = df.copy()

print(
    "Raw-data copy created:",
    df_raw.shape
)

Raw-data copy created:


In [15]:
# Inspect record counts and the original timestamp range.
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nFirst timestamp:")
print(df["HourEnding"].min())

print("\nLast timestamp:")
print(df["HourEnding"].max())

Number of rows:
Number of columns:

First timestamp:
01/01/2019 01:00

Last timestamp:
12/31/2019 24:00


In [16]:
# Inspect the source HourEnding representation.
print(
    "HourEnding data type:",
    df["HourEnding"].dtype
)

HourEnding data type:


In [17]:
# Attempt initial timestamp parsing to diagnose unsupported source formats.
df["HourEnding"] = pd.to_datetime(
    df["HourEnding"],
    errors="coerce"
)

invalid_timestamps = df["HourEnding"].isna().sum()

print(
    "Invalid timestamps:",
    invalid_timestamps
)

Invalid timestamps:


In [18]:
# Check the initial parsed timestamp ordering.
print(
    "Timestamps are sorted:",
    df["HourEnding"].is_monotonic_increasing
)

Timestamps are sorted:


In [19]:
# Count exact duplicate source rows.
exact_duplicate_count = df.duplicated().sum()

print(
    "Exact duplicate rows:",
    exact_duplicate_count
)

Exact duplicate rows:


In [20]:
# Inspect duplicate timestamps before addressing parsing artifacts.
duplicate_time_mask = df.duplicated(
    subset=["HourEnding"],
    keep=False
)

duplicate_time_count = duplicate_time_mask.sum()

print(
    "Rows with duplicate timestamps:",
    duplicate_time_count
)

Rows with duplicate timestamps:


In [21]:
# Inspect observations that share an initially parsed timestamp.
duplicate_times = (
    df.loc[duplicate_time_mask]
    .sort_values("HourEnding")
)

duplicate_times.head(20)

,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
23,NaT,9170.205495,1277.553214,3361.038342,965.196812,14422.313391,3000.414658,6849.033113,1449.080820,40494.835846
47,NaT,10113.377064,1441.306104,3398.570525,978.095296,15384.915820,3855.397030,7608.084317,1507.680708,44287.426865
71,NaT,11103.183304,1524.394565,3215.751345,922.314720,15044.509932,3770.851749,7211.111135,1399.748593,44191.865344
95,NaT,10105.665274,1414.573228,3138.253968,831.302075,12753.155654,3036.327114,6095.948147,1261.698040,38636.923501
119,NaT,9395.859220,1196.479093,3061.631659,736.273727,10803.396960,2725.644503,5445.849500,1151.415258,34516.549921
143,NaT,8801.624832,1033.847462,2983.423519,666.303793,9642.153177,2593.429152,4717.908150,1006.253951,31444.944036
167,NaT,8990.360194,1058.559995,3100.628434,674.618990,9779.668712,2670.119318,4860.974709,1053.245797,32188.176150
191,NaT,8695.438280,1098.448227,3094.092046,764.813395,10938.476574,2655.822788,4768.471775,1078.074959,33093.638043
215,NaT,9383.645700,1345.199469,3070.947751,786.233406,11860.893129,2653.971995,5534.586578,1118.482611,35753.960640
239,NaT,9439.754807,1311.555846,3030.530595,787.179337,12063.895974,2618.535853,5479.898734,1093.400464,35824.751611


In [22]:
# Summarize differences between consecutive parsed timestamps.
df["time_difference"] = df["HourEnding"].diff()

time_gap_summary = (
    df["time_difference"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "Time-interval distribution:"
)

time_gap_summary

Time-interval distribution:


time_difference
0 days 01:00:00    8027
0 days 02:00:00       1
NaT                 732
Name: count, dtype: int64

In [23]:
# Inspect non-hourly intervals from the initial parsing attempt.
unusual_intervals = df[
    df["time_difference"].notna()
    & (df["time_difference"] != pd.Timedelta(hours=1))
]

print(
    "Unusual intervals:",
    len(unusual_intervals)
)

unusual_intervals[
    ["HourEnding", "time_difference"]
].head(20)

Unusual intervals:


,HourEnding,time_difference
1634,2019-03-10 04:00:00,0 days 02:00:00


In [24]:
# Inspect the distribution of regional and system load values.
load_columns = [
    "COAST",
    "EAST",
    "FWEST",
    "NORTH",
    "NCENT",
    "SOUTH",
    "SCENT",
    "WEST",
    "ERCOT",
]

load_summary = df[load_columns].describe().T

load_summary

,count,mean,std,min,25%,50%,75%,max
COAST,8760.0,12354.094953,2808.094389,7913.343226,10321.588850,11554.269809,13911.918240,21256.115190
EAST,8760.0,1485.166996,332.496152,853.665800,1251.346266,1403.672738,1675.482935,2554.015379
FWEST,8760.0,3435.597449,301.126593,2871.475030,3173.360034,3418.315688,3659.718802,4307.552925
NORTH,8760.0,854.125749,177.913390,547.377960,725.198803,815.655079,951.837606,1476.329919
NCENT,8760.0,13819.153682,3592.068404,7851.374507,11294.865534,12902.004111,15739.893931,25493.791364
SOUTH,8760.0,3607.938677,880.491237,2127.374424,2947.129271,3391.237505,4179.690931,6039.713459
SCENT,8760.0,6970.015405,1864.466114,3979.759856,5635.780006,6473.156980,7949.690186,12784.857878
WEST,8760.0,1291.835067,238.693083,879.438391,1121.547443,1229.720539,1412.856788,2117.399078
ERCOT,8760.0,43817.927977,9767.889899,27612.054044,37019.256240,41071.215074,48936.649169,74665.579486


In [25]:
# Count zero and negative load values.
nonpositive_counts = (
    df[load_columns] <= 0
).sum()

print(
    "Zero or negative values:"
)

nonpositive_counts

Zero or negative values:


COAST    0
EAST     0
FWEST    0
NORTH    0
NCENT    0
SOUTH    0
SCENT    0
WEST     0
ERCOT    0
dtype: int64

In [26]:
# Reconcile the reported ERCOT load with summed weather-zone loads.
zone_columns = [
    "COAST",
    "EAST",
    "FWEST",
    "NORTH",
    "NCENT",
    "SOUTH",
    "SCENT",
    "WEST",
]

df["calculated_ercot_mw"] = (
    df[zone_columns].sum(axis=1)
)

df["ercot_difference_mw"] = (
    df["ERCOT"]
    - df["calculated_ercot_mw"]
)

print(
    "Reported minus calculated ERCOT load (MW)"
)
df["ercot_difference_mw"].describe()

Reported minus calculated ERCOT load (MW)


count    8.760000e+03
mean     5.022829e-08
std      8.216932e-07
min     -3.000008e-06
25%     -7.275958e-12
50%      0.000000e+00
75%      9.999931e-07
max      3.000008e-06
Name: ercot_difference_mw, dtype: float64

In [27]:
# Check the maximum absolute zone-to-system discrepancy.
maximum_absolute_difference = (
    df["ercot_difference_mw"]
    .abs()
    .max()
)

print(
    "Maximum absolute difference (MW): ",
    maximum_absolute_difference
)

Maximum absolute difference (MW):


In [ ]:
# Summarize initial data-quality diagnostics.
quality_summary = pd.DataFrame({
    "metric_en": [
        "Row count",
        "Column count",
        "Missing values",
        "Invalid timestamps",
        "Exact duplicate rows",
        "Rows with duplicate timestamps",
        "Unusual time intervals",
        "Nonpositive load values",
        "Maximum zone-total difference (MW)",
    ],
    "metric_description": [
        "Total observations",
        "Columns in source dataset",
        "Missing entries across original columns",
        "Timestamps not parsed in initial conversion",
        "Exact duplicate records",
        "Rows sharing initial parsed time labels",
        "Non-hourly differences in initial timestamps",
        "Zero or negative load entries",
        "Largest difference between system and zone totals (MW)",
    ],
    "value": [
        len(df),
        len(df_raw.columns),
        int(df_raw.isna().sum().sum()),
        int(invalid_timestamps),
        int(exact_duplicate_count),
        int(duplicate_time_count),
        int(len(unusual_intervals)),
        int((df[load_columns] <= 0).sum().sum()),
        float(maximum_absolute_difference),
    ],
})

quality_summary

In [29]:
# Inspect source strings corresponding to failed initial timestamps.
failed_mask = df["HourEnding"].isna()

failed_raw_values = df_raw.loc[
    failed_mask,
    "HourEnding"
]

print(
    "Failed original timestamps: ",
    len(failed_raw_values)
)

failed_raw_values.head(10)

Failed original timestamps:


23     01/01/2019 24:00
47     01/02/2019 24:00
71     01/03/2019 24:00
95     01/04/2019 24:00
119    01/05/2019 24:00
143    01/06/2019 24:00
167    01/07/2019 24:00
191    01/08/2019 24:00
215    01/09/2019 24:00
239    01/10/2019 24:00
Name: HourEnding, dtype: str

In [30]:
failed_raw_values.tail(10)

8543    12/22/2019 24:00
8567    12/23/2019 24:00
8591    12/24/2019 24:00
8615    12/25/2019 24:00
8639    12/26/2019 24:00
8663    12/27/2019 24:00
8687    12/28/2019 24:00
8711    12/29/2019 24:00
8735    12/30/2019 24:00
8759    12/31/2019 24:00
Name: HourEnding, dtype: str

In [31]:
failed_raw_values.astype(str).str[-8:].value_counts()

HourEnding
19 24:00    365
2:00 DST      1
Name: count, dtype: int64

In [32]:
# Identify failed timestamp strings other than the 24:00 convention.
non_24_failed_values = failed_raw_values[
    ~failed_raw_values
    .astype(str)
    .str.strip()
    .str.endswith("24:00")
]

print(
    "Failed values other than 24:00: "
)

non_24_failed_values

Failed values other than 24:00:


7345    11/03/2019 02:00 DST
Name: HourEnding, dtype: str

In [33]:
# Restart from the untouched source and preserve the original time labels.
df = df_raw.copy()

df["HourEnding_raw"] = (
    df["HourEnding"]
    .astype(str)
    .str.strip()
)

print(
    "Raw timestamp column preserved: "
)

Raw timestamp column preserved:


In [34]:
# Capture the explicit daylight-saving suffix.
df["is_dst_label"] = (
    df["HourEnding_raw"]
    .str.endswith(" DST")
)

print(
    "Rows with DST label: ",
    df["is_dst_label"].sum()
)

Rows with DST label:


In [ ]:
# Remove the DST suffix for parsing while preserving the indicator.
normalized_timestamp_text = (
    df["HourEnding_raw"]
    .str.replace(
        r"\s+DST$",
        "",
        regex=True
    )
)

In [36]:
# Identify source Hour Ending values using 24:00.
is_hour_24 = (
    normalized_timestamp_text
    .str.endswith("24:00")
)

print(
    "24:00 records:",
    is_hour_24.sum()
)

24:00 records:


In [ ]:
# Temporarily replace 24:00 with 00:00 to allow parsing.
normalized_timestamp_text = (
    normalized_timestamp_text
    .str.replace(
        r"\s24:00$",
        " 00:00",
        regex=True
    )
)

In [ ]:
# Parse the normalized local-clock labels.
parsed_local_time = pd.to_datetime(
    normalized_timestamp_text,
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

In [ ]:
# Advance 24:00 records to midnight of the following day.
parsed_local_time.loc[is_hour_24] = (
    parsed_local_time.loc[is_hour_24]
    + pd.Timedelta(days=1)
)

In [ ]:
# Save the normalized local-clock timestamp as a separate field.
df["HourEnding_local"] = parsed_local_time

In [41]:
# Verify that no unparsed timestamps remain.
remaining_invalid_timestamps = (
    df["HourEnding_local"]
    .isna()
    .sum()
)

print(
    "Remaining invalid timestamps: ",
    remaining_invalid_timestamps
)

Remaining invalid timestamps:


In [42]:
# Inspect records affected by 24:00 and DST normalization.
df.loc[
    is_hour_24 | df["is_dst_label"],
    [
        "HourEnding_raw",
        "HourEnding_local",
        "is_dst_label",
    ]
].head()

,HourEnding_raw,HourEnding_local,is_dst_label
23,01/01/2019 24:00,2019-01-02,False
47,01/02/2019 24:00,2019-01-03,False
71,01/03/2019 24:00,2019-01-04,False
95,01/04/2019 24:00,2019-01-05,False
119,01/05/2019 24:00,2019-01-06,False


In [43]:
# Inspect explicitly labeled DST records.
df.loc[
    df["is_dst_label"],
    [
        "HourEnding_raw",
        "HourEnding_local",
        "is_dst_label",
        "ERCOT",
    ]
]

,HourEnding_raw,HourEnding_local,is_dst_label,ERCOT
7345,11/03/2019 02:00 DST,2019-11-03 02:00:00,True,33309.960879


In [44]:
# Recheck duplicate local-clock labels after normalization.
duplicate_local_mask = df.duplicated(
    subset=["HourEnding_local"],
    keep=False
)

duplicate_local_times = (
    df.loc[
        duplicate_local_mask,
        [
            "HourEnding_raw",
            "HourEnding_local",
            "is_dst_label",
            "ERCOT",
        ]
    ]
    .sort_values("HourEnding_local")
)

print(
    "Rows with duplicate local timestamps: ",
    len(duplicate_local_times)
)

duplicate_local_times

Rows with duplicate local timestamps:


,HourEnding_raw,HourEnding_local,is_dst_label,ERCOT
7344,11/03/2019 02:00,2019-11-03 02:00:00,False,33324.971467
7345,11/03/2019 02:00 DST,2019-11-03 02:00:00,True,33309.960879


In [45]:
# Summarize local-clock time differences, including DST transitions.
df["time_difference_local"] = (
    df["HourEnding_local"]
    .diff()
)

time_interval_distribution = (
    df["time_difference_local"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "Local-time interval distribution: "
)

time_interval_distribution

Local-time interval distribution:


time_difference_local
0 days 00:00:00       1
0 days 01:00:00    8757
0 days 02:00:00       1
NaT                   1
Name: count, dtype: int64

In [ ]:
# Summarize final data-quality diagnostics.
final_duplicate_local_count = (
    df.duplicated(
        subset=["HourEnding_local"],
        keep=False
    )
    .sum()
)

non_one_hour_intervals = df[
    df["time_difference_local"].notna()
    & (
        df["time_difference_local"]
        != pd.Timedelta(hours=1)
    )
]

final_quality_summary = pd.DataFrame({
    "metric_en": [
        "Row count",
        "Original column count",
        "Missing load values",
        "Unparsed timestamps after normalization",
        "Exact duplicate rows in source data",
        "Rows sharing a local timestamp",
        "Non-one-hour local-clock intervals",
        "Nonpositive load values",
        "Maximum zone-total difference (MW)",
    ],
    "metric_description": [
        "Total observations",
        "Original source columns",
        "Missing load entries",
        "Unparsed timestamps after normalization",
        "Exact duplicates in source data",
        "Rows sharing a normalized local-clock label",
        "Non-hourly local-clock intervals",
        "Zero or negative load entries",
        "Largest difference between system and zone totals (MW)",
    ],
    "value": [
        len(df),
        len(df_raw.columns),
        int(df_raw[load_columns].isna().sum().sum()),
        int(df["HourEnding_local"].isna().sum()),
        int(df_raw.duplicated().sum()),
        int(final_duplicate_local_count),
        int(len(non_one_hour_intervals)),
        int((df[load_columns] <= 0).sum().sum()),
        float(maximum_absolute_difference),
    ],
})

final_quality_summary

## 3. Preliminary Findings and Timestamp Diagnosis

### Dataset completeness and load quality

- The 2019 source contains **8,760 hourly observations** and 10 original fields: Hour Ending, eight weather-zone loads, and an ERCOT system total.
- The source columns have no missing values, zero or negative loads, or exact duplicate rows.
- The zone-load sum agrees with the reported ERCOT total within floating-point precision (maximum absolute difference approximately **0.000003 MW**).

### Why initial timestamp checks appeared to fail

- An initial Pandas datetime conversion yielded **366 unparsed timestamps**. These are formatting artifacts, not evidence of 366 missing measurements.
- **365** records use the industry-specific `24:00` Hour Ending notation. For example, `01/01/2019 24:00` corresponds to `01/02/2019 00:00`.
- One record, `11/03/2019 02:00 DST`, contains an explicit daylight-saving label that identifies an autumn clock-change observation.
- Converting these strings to `NaT` made the initial duplicate-timestamp check misleading. The apparent duplicates must be re-evaluated after normalization rather than deleted.

### Normalization and local-clock validation

- Preserve `HourEnding_raw`, normalize `24:00` to the next day's midnight, and retain the explicit DST indicator in `is_dst_label`.
- After normalization, there are **no unparsed timestamps**. During the fall transition, two different observations share the same local-clock label but have different load values and DST labels.
- The local-time interval distribution contains **8,757 one-hour differences**, one zero-hour difference (fall clock repetition), and one two-hour difference (spring clock advance), plus the expected initial `NaT` difference.
- These are local-clock effects, not justification for filling in or deleting real hourly observations.

**Assessment:** The 2019 load values appear complete, plausible, and internally consistent after the source time conventions are accounted for. A separate monotonic UTC key is needed for subsequent merges and time-series modeling.

## 4. Construction of a Unique UTC Timeline

During the autumn daylight-saving transition, normalized local timestamps can repeat, so they are not suitable as unique keys. Since the input contains the expected number of ordered hourly observations, this analysis constructs a continuous UTC Hour Ending sequence from the first known timestamp and the original row order.

**Assumption:** Source records must be complete and chronologically ordered. The generated UTC timeline should be checked against the original Hour Ending labels, especially around daylight-saving transitions.

In [49]:
# Create a stable source-row interval identifier.
df["interval_id"] = range(1, len(df) + 1)

print(
    "Interval ID is unique: ",
    df["interval_id"].is_unique
)

print(
    "Interval ID range:",
    df["interval_id"].min(),
    "to",
    df["interval_id"].max()
)


Interval ID is unique:
Interval ID range:


In [50]:
# Convert the first known local Hour Ending to UTC using the America/Chicago timezone.
first_local_hour_ending = pd.Timestamp(
    "2019-01-01 01:00:00"
)

first_utc_hour_ending = (
    first_local_hour_ending
    .tz_localize("America/Chicago")
    .tz_convert("UTC")
)

print(
    "First local Hour Ending: ",
    first_local_hour_ending
)

print(
    "First UTC Hour Ending: ",
    first_utc_hour_ending
)

First local Hour Ending:
First UTC Hour Ending:


In [ ]:
# Generate a continuous hourly UTC sequence from the first Hour Ending.
df["HourEnding_utc"] = pd.date_range(
    start=first_utc_hour_ending,
    periods=len(df),
    freq="h",
)

In [52]:
# Verify uniqueness, ordering, and bounds of the constructed UTC sequence.
print(
    "UTC data type:",
    df["HourEnding_utc"].dtype
)

print(
    "UTC timestamps are unique: ",
    df["HourEnding_utc"].is_unique
)

print(
    "UTC timestamps are sorted: ",
    df["HourEnding_utc"].is_monotonic_increasing
)

print(
    "First UTC timestamp:",
    df["HourEnding_utc"].min()
)

print(
    "Last UTC timestamp:",
    df["HourEnding_utc"].max()
)

UTC data type:
UTC timestamps are unique:
UTC timestamps are sorted:
First UTC timestamp:
Last UTC timestamp:


In [53]:
# Inspect intervals between consecutive UTC timestamps.
df["time_difference_utc"] = (
    df["HourEnding_utc"]
    .diff()
)

utc_interval_distribution = (
    df["time_difference_utc"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "UTC interval distribution: "
)

utc_interval_distribution

UTC interval distribution:


time_difference_utc
0 days 01:00:00    8759
NaT                   1
Name: count, dtype: int64

In [54]:
# Compare raw, local, and UTC timestamps near the autumn DST transition.
dst_position = df.index[
    df["is_dst_label"]
][0]

dst_comparison = df.loc[
    dst_position - 3 : dst_position + 3,
    [
        "interval_id",
        "HourEnding_raw",
        "HourEnding_local",
        "is_dst_label",
        "HourEnding_utc",
        "ERCOT",
    ],
]

dst_comparison

,interval_id,HourEnding_raw,HourEnding_local,is_dst_label,HourEnding_utc,ERCOT
7342,7343,11/02/2019 24:00,2019-11-03 00:00:00,False,2019-11-03 05:00:00+00:00,34173.503870
7343,7344,11/03/2019 01:00,2019-11-03 01:00:00,False,2019-11-03 06:00:00+00:00,33589.391457
7344,7345,11/03/2019 02:00,2019-11-03 02:00:00,False,2019-11-03 07:00:00+00:00,33324.971467
7345,7346,11/03/2019 02:00 DST,2019-11-03 02:00:00,True,2019-11-03 08:00:00+00:00,33309.960879
7346,7347,11/03/2019 03:00,2019-11-03 03:00:00,False,2019-11-03 09:00:00+00:00,33513.366296
7347,7348,11/03/2019 04:00,2019-11-03 04:00:00,False,2019-11-03 10:00:00+00:00,33868.252948
7348,7349,11/03/2019 05:00,2019-11-03 05:00:00,False,2019-11-03 11:00:00+00:00,34609.838890


In [55]:
# Inspect the spring DST transition.
spring_dst_window = df[
    df["HourEnding_local"].between(
        "2019-03-10 00:00:00",
        "2019-03-10 06:00:00",
    )
][
    [
        "interval_id",
        "HourEnding_raw",
        "HourEnding_local",
        "HourEnding_utc",
        "ERCOT",
    ]
]

spring_dst_window

,interval_id,HourEnding_raw,HourEnding_local,HourEnding_utc,ERCOT
1631,1632,03/09/2019 24:00,2019-03-10 00:00:00,2019-03-10 06:00:00+00:00,32867.372225
1632,1633,03/10/2019 01:00,2019-03-10 01:00:00,2019-03-10 07:00:00+00:00,31364.579509
1633,1634,03/10/2019 02:00,2019-03-10 02:00:00,2019-03-10 08:00:00+00:00,30301.323301
1634,1635,03/10/2019 04:00,2019-03-10 04:00:00,2019-03-10 09:00:00+00:00,29634.884625
1635,1636,03/10/2019 05:00,2019-03-10 05:00:00,2019-03-10 10:00:00+00:00,29392.270031
1636,1637,03/10/2019 06:00,2019-03-10 06:00:00,2019-03-10 11:00:00+00:00,29669.755886


In [ ]:
# Document how each time field is used downstream.
time_field_summary = pd.DataFrame({
    "field": [
        "HourEnding_raw",
        "HourEnding_local",
        "HourEnding_utc",
        "interval_id",
    ],
    "purpose_en": [
        "Preserve the original ERCOT source label",
        "Human-readable local-clock analysis and charts",
        "Unique key for joins, SQL, and time-series modeling",
        "Stable row-order identifier",
    ],
    "purpose_description": [
        "Original ERCOT Hour Ending label",
        "Local clock interpretation and plotting",
        "Unique join/SQL/time-series key",
        "Stable identifier following source row order",
    ],
})

time_field_summary

### UTC Timeline Validation

- Assign a unique `interval_id` from 1 through 8,760 following ERCOT's original row order.
- `01/01/2019 01:00` Central Standard Time corresponds to `2019-01-01 07:00 UTC`.
- Generate one UTC timestamp per row at an hourly frequency; the resulting UTC index is unique, ordered, and hourly continuous.
- Unlike local clock labels, the constructed UTC timeline has no apparent skipped or repeated time labels across DST transitions.
- Use `HourEnding_utc` for joins, SQL keys, and forecasting; retain `HourEnding_local` for interpretation and local-time plots.

The UTC check validates the **constructed sequence**; it does not independently prove that every source Hour Ending label is correctly ordered.

In [58]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cleaning.clean_ercot_load import (
    clean_ercot_hourly_load,
    validate_ercot_hourly_load,
)
print("Import successful")

Import successful


In [ ]:
import pandas as pd

files_2019 = [
    file
    for file in excel_files
    if "2019" in file.name
]

print("2019 files found:", [p.relative_to(project_root).as_posix() for p in files_2019])

In [60]:
cleaned_2019 = clean_ercot_hourly_load(
    file_path=files_2019[0],
    year=2019,
)

validation_2019 = validate_ercot_hourly_load(
    df=cleaned_2019,
    year=2019,
)

validation_2019_table = pd.DataFrame(
    validation_2019.items(),
    columns=["metric", "value"],
)

validation_2019_table

,metric,value
0,year,2019
1,row_count,8760
2,expected_hours,8760
3,row_count_matches,True
4,missing_load_values,0
5,unparsed_local_timestamps,0
6,unique_utc_timestamps,True
7,utc_is_sorted,True
8,non_hourly_utc_intervals,0
9,exact_duplicate_source_rows,0


In [61]:
# Spot-check the DST rows after applying the reusable cleaner.
cleaned_2019.loc[
    cleaned_2019["is_dst_label"]
    | cleaned_2019["HourEnding_local"].duplicated(keep=False),
    [
        "HourEnding_raw",
        "HourEnding_local",
        "HourEnding_utc",
        "is_dst_label",
        "ERCOT",
    ],
]

,HourEnding_raw,HourEnding_local,HourEnding_utc,is_dst_label,ERCOT
7344,11/03/2019 02:00,2019-11-03 02:00:00,2019-11-03 07:00:00+00:00,False,33324.971467
7345,11/03/2019 02:00 DST,2019-11-03 02:00:00,2019-11-03 08:00:00+00:00,True,33309.960879


## 5. Batch Cleaning and Validation (2019–2025)

Apply the reusable cleaning and validation functions to each available annual load file, preserving per-year data-quality metrics.

In [63]:
cleaned_by_year = {}
validation_results = []

for year in range(2019, 2026):
    matching_files = [
        file
        for file in excel_files
        if str(year) in file.name
    ]

    if len(matching_files) != 1:
        print(
            f"{year}: expected 1 file, "
            f"found {len(matching_files)}"
        )
        continue

    try:
        cleaned_df = clean_ercot_hourly_load(
            file_path=matching_files[0],
            year=year,
        )

        validation = validate_ercot_hourly_load(
            df=cleaned_df,
            year=year,
        )

        cleaned_by_year[year] = cleaned_df
        validation_results.append(validation)

        print(f"{year}: completed")

    except Exception as error:
        print(f"{year}: failed — {error}")

2019: completed
2020: completed
2021: completed
2022: completed
2023: completed
2024: completed
2025: completed


In [64]:
for year in range(2019, 2026):
    matching_files = [
        file
        for file in excel_files
        if str(year) in file.name
    ]

    print(f"\n===== {year} =====")

    for file in matching_files:
        print("File:", file.name)

        preview = pd.read_excel(
            file,
            nrows=3,
        )

        print("Columns:")
        print([repr(column) for column in preview.columns])

        display(preview)


===== 2019 =====
File: Native_Load_2019.xlsx
Columns:
["'HourEnding'", "'COAST'", "'EAST'", "'FWEST'", "'NORTH'", "'NCENT'", "'SOUTH'", "'SCENT'", "'WEST'", "'ERCOT'"]


,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2019 01:00,9783.594839,1264.194059,3164.720629,827.759428,11697.766093,3066.891146,5993.974289,1282.542956,37081.443439
1,01/01/2019 02:00,9726.062326,1270.494294,3178.950849,830.256302,11787.936174,3140.383209,6029.773970,1295.132811,37258.989933
2,01/01/2019 03:00,9654.373521,1269.504203,3200.881160,831.505553,11861.628382,3137.721342,6043.348512,1301.225133,37300.187809



===== 2020 =====
File: Native_Load_2020.xlsx
Columns:
["'HourEnding'", "'COAST'", "'EAST'", "'FWEST'", "'NORTH'", "'NCENT'", "'SOUTH'", "'SCENT'", "'WEST'", "'ERCOT'"]


,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2020 01:00,9890.091522,1364.278198,3721.972022,773.878822,11719.863767,2707.841517,5473.246071,1172.943179,36824.115099
1,01/01/2020 02:00,9751.586415,1369.569694,3735.514034,775.063743,11639.455987,2706.844837,5387.380467,1165.951313,36531.366488
2,01/01/2020 03:00,9603.421087,1373.528720,3735.851866,774.153775,11515.421981,2650.378864,5270.331958,1149.076769,36072.165020



===== 2021 =====
File: Native_Load_2021.xlsx
Columns:
["'Hour Ending'", "'COAST'", "'EAST'", "'FWEST'", "'NORTH'", "'NCENT'", "'SOUTH'", "'SCENT'", "'WEST'", "'ERCOT'"]


...

,Hour Ending,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2023 01:00,8514.637876,1218.506271,5073.526557,917.394049,9896.757586,2859.017922,6012.400469,1116.933155,35609.173887
1,01/01/2023 02:00,8345.632899,1179.885576,5016.774637,906.854582,9675.869720,2845.779210,5934.084260,1107.417972,35012.298856
2,01/01/2023 03:00,8125.706879,1151.144653,5016.303220,894.817013,9490.332365,2734.163642,5855.790279,1101.324245,34369.582295



===== 2024 =====
File: Native_Load_2024.xlsx
Columns:
["'Hour Ending'", "'COAST'", "'EAST'", "'FWEST'", "'NORTH'", "'NCENT'", "'SOUTH'", "'SCENT'", "'WEST'", "'ERCOT'"]


,Hour Ending,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2024 01:00,10484.137154,1375.738368,6210.082088,1250.403165,11799.512618,2929.799758,6020.385247,1153.421779,41223.480177
1,01/01/2024 02:00,10365.816869,1455.235587,6223.669226,1272.979716,12094.701362,2916.987913,5987.545519,1162.463555,41479.399746
2,01/01/2024 03:00,10206.968091,1396.973111,6270.219255,1290.712338,12414.588720,2835.801915,5874.877381,1182.629316,41472.770126



===== 2025 =====
File: Native_Load_2025.xlsx
Columns:
["'Hour Ending'", "'COAST'", "'EAST'", "'FWEST'", "'NORTH'", "'NCENT'", "'SOUTH'", "'SCENT'", "'WEST'", "'ERCOT'"]


,Hour Ending,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2025 01:00,10538.553193,1545.465863,7090.619810,1388.565588,11964.866264,3012.769998,6447.297871,1235.721317,43223.859905
1,01/01/2025 02:00,10477.912860,1553.994477,7040.460607,1390.392647,12064.919506,2962.803597,6382.650858,1249.397659,43122.532212
2,01/01/2025 03:00,10363.015361,1534.575620,7028.741719,1390.634577,12040.262276,2981.433181,6320.119991,1259.729121,42918.511847


## 6. Multi-Year Validation Summary

Review cross-year quality metrics and investigate the reported ERCOT system total versus the sum of the eight weather-zone loads, with particular attention to the flagged 2021 discrepancy period.

In [66]:
validation_summary = (
    pd.DataFrame(validation_results)
    .set_index("year")
    .sort_index()
)

validation_summary.T

year,2019,2020,2021,2022,2023,2024,2025
row_count,8760,8784,8760,8760,8760,8784,8760
expected_hours,8760,8784,8760,8760,8760,8784,8760
row_count_matches,True,True,True,True,True,True,True
missing_load_values,0,0,0,0,0,0,0
unparsed_local_timestamps,0,0,0,0,0,0,0
unique_utc_timestamps,True,True,True,True,True,True,True
utc_is_sorted,True,True,True,True,True,True,True
non_hourly_utc_intervals,0,0,0,0,0,0,0
exact_duplicate_source_rows,0,0,0,0,0,0,0
nonpositive_load_values,0,0,0,0,0,0,0


In [67]:
from src.cleaning.clean_ercot_load import ZONE_COLUMNS

print(ZONE_COLUMNS)

['COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST']


In [68]:
# Inspect the largest ERCOT-versus-zone-sum discrepancies in 2021.
df_2021 = cleaned_by_year[2021].copy()

largest_2021_differences = (
    df_2021.loc[
        :,
        [
            "HourEnding_raw",
            "HourEnding_local",
            "HourEnding_utc",
            *ZONE_COLUMNS,
            "ERCOT",
            "calculated_ercot_mw",
            "ercot_difference_mw",
        ],
    ]
    .assign(
        absolute_difference_mw=lambda df:
        df["ercot_difference_mw"].abs()
    )
    .sort_values(
        "absolute_difference_mw",
        ascending=False,
    )
    .head(20)
)

largest_2021_differences

,HourEnding_raw,HourEnding_local,HourEnding_utc,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT,calculated_ercot_mw,ercot_difference_mw,absolute_difference_mw
3281,05/17/2021 19:00,2021-05-17 19:00:00,2021-05-18 00:00:00+00:00,13016.792545,1590.998950,3962.536858,814.049267,14028.298907,4406.681694,8192.418314,1391.379082,49762.986886,47403.155617,2359.831269,2359.831269
3280,05/17/2021 18:00,2021-05-17 18:00:00,2021-05-17 23:00:00+00:00,13681.793669,1594.334954,3989.987158,827.846902,13995.632588,4596.844010,8039.647947,1411.522766,50349.531758,48137.609994,2211.921764,2211.921764
3282,05/17/2021 20:00,2021-05-17 20:00:00,2021-05-18 01:00:00+00:00,12596.149948,1577.226696,3932.037713,782.483971,13614.527885,4223.046008,7906.019978,1313.469440,48070.553923,45944.961639,2125.592284,2125.592284
3283,05/17/2021 21:00,2021-05-17 21:00:00,2021-05-18 02:00:00+00:00,12423.331460,1569.735996,3879.492834,762.804805,13433.352011,4181.756018,7766.279033,1277.021857,47199.646632,45293.774014,1905.872618,1905.872618
3279,05/17/2021 17:00,2021-05-17 17:00:00,2021-05-17 22:00:00+00:00,14884.841938,1678.612349,4052.477923,829.829908,13618.414678,4723.100992,7629.274507,1398.354593,50608.643284,48814.906888,1793.736396,1793.736396
3284,05/17/2021 22:00,2021-05-17 22:00:00,2021-05-18 03:00:00+00:00,12178.379035,1506.864038,3849.464864,730.612831,13020.926414,4080.163253,7595.986335,1229.469433,45975.766292,44191.866203,1783.900089,1783.900089
3275,05/17/2021 13:00,2021-05-17 13:00:00,2021-05-17 18:00:00+00:00,15159.819779,1581.293066,3823.716480,729.510729,12422.488581,4292.698571,7654.386381,1184.652381,48516.275323,46848.565968,1667.709355,1667.709355
3299,05/18/2021 13:00,2021-05-18 13:00:00,2021-05-18 18:00:00+00:00,13576.463340,1683.773998,3714.652530,659.006771,11471.469035,3933.699519,7759.309731,1061.464687,45517.252626,43859.839611,1657.413015,1657.413015
3298,05/18/2021 12:00,2021-05-18 12:00:00,2021-05-18 17:00:00+00:00,13203.845418,1588.181954,3662.587297,656.557443,11535.065292,3706.789125,7408.366742,1060.759738,44474.220964,42822.153009,1652.067955,1652.067955
3274,05/17/2021 12:00,2021-05-17 12:00:00,2021-05-17 17:00:00+00:00,14508.917986,1560.495656,3776.188053,702.495445,12007.210007,4052.480350,7405.541200,1165.452996,46737.156343,45178.781693,1558.374650,1558.374650


In [69]:
# Count and inspect records above the material discrepancy threshold.
difference_threshold_mw = 0.01

large_difference_rows_2021 = df_2021.loc[
    df_2021["ercot_difference_mw"].abs()
    > difference_threshold_mw
].copy()

print(
    "Rows above threshold:",
    len(large_difference_rows_2021),
)

large_difference_rows_2021[
    [
        "HourEnding_raw",
        "HourEnding_local",
        "ERCOT",
        "calculated_ercot_mw",
        "ercot_difference_mw",
    ]
]

Rows above threshold: 48


,HourEnding_raw,HourEnding_local,ERCOT,calculated_ercot_mw,ercot_difference_mw
3263,05/17/2021 01:00,2021-05-17 01:00:00,36961.979842,36108.827384,853.152458
3264,05/17/2021 02:00,2021-05-17 02:00:00,35467.187055,34699.478829,767.708226
3265,05/17/2021 03:00,2021-05-17 03:00:00,34603.949944,33879.583141,724.366803
3266,05/17/2021 04:00,2021-05-17 04:00:00,34203.922200,33490.479179,713.443021
3267,05/17/2021 05:00,2021-05-17 05:00:00,34594.242757,33827.722440,766.520317
3268,05/17/2021 06:00,2021-05-17 06:00:00,36036.192765,35159.305164,876.887601
3269,05/17/2021 07:00,2021-05-17 07:00:00,38343.509273,37304.079220,1039.430053
3270,05/17/2021 08:00,2021-05-17 08:00:00,39938.493484,38785.481116,1153.012368
3271,05/17/2021 09:00,2021-05-17 09:00:00,41614.973313,40406.011399,1208.961914
3272,05/17/2021 10:00,2021-05-17 10:00:00,43235.284580,41905.270117,1330.014463


In [70]:
df_2021["ercot_difference_mw"].describe(
    percentiles=[
        0.01,
        0.25,
        0.50,
        0.75,
        0.99,
    ]
)

count    8.760000e+03
mean     7.035498e+00
std      9.957036e+01
min     -3.000001e-06
1%      -2.000001e-06
25%     -7.275958e-12
50%      0.000000e+00
75%      9.999931e-07
99%      2.000004e-06
max      2.359831e+03
Name: ercot_difference_mw, dtype: float64

In [71]:
maximum_allowed_difference_mw = 0.01

validation_checks = pd.DataFrame({
    "year": validation_summary.index,
    "passed": (
        validation_summary["row_count_matches"]
        & (validation_summary["missing_load_values"] == 0)
        & (
            validation_summary[
                "unparsed_local_timestamps"
            ] == 0
        )
        & validation_summary["unique_utc_timestamps"]
        & validation_summary["utc_is_sorted"]
        & (
            validation_summary[
                "non_hourly_utc_intervals"
            ] == 0
        )
        & (
            validation_summary[
                "exact_duplicate_source_rows"
            ] == 0
        )
        & (
            validation_summary[
                "nonpositive_load_values"
            ] == 0
        )
        & (
            validation_summary[
                "maximum_zone_difference_mw"
            ] <= maximum_allowed_difference_mw
        )
    ).to_numpy(),
}).set_index("year")

validation_checks

,passed
year,
2019,True
2020,True
2021,False
2022,True
2023,True
2024,True
2025,True


### 2021 Regional Aggregation Discrepancy

- **48 records** are flagged, concentrated around **May 17–18, 2021**.
- Their ERCOT-minus-zone-sum differences are materially larger than floating-point rounding discrepancies.
- The remainder of the annual difference distribution is near zero (including the median and 99th percentile).
- The anomaly is localized; it does not on its own imply that the load-cleaning arithmetic is broadly incorrect.

The affected rows are retained and flagged for downstream quality handling, rather than silently overwritten.

In [73]:
# Locate the exact 2021 mismatch window.
bad_2021 = (
    cleaned_by_year[2021]
    .loc[
        cleaned_by_year[2021][
            "ercot_difference_mw"
        ].abs() > 0.01
    ]
    .sort_values("HourEnding_utc")
    .copy()
)

print("Number of affected rows:", len(bad_2021))
print(
    "First affected time:",
    bad_2021["HourEnding_raw"].iloc[0],
)
print(
    "Last affected time:",
    bad_2021["HourEnding_raw"].iloc[-1],
)
print(
    "Minimum difference (MW):",
    bad_2021["ercot_difference_mw"].min(),
)
print(
    "Maximum difference (MW):",
    bad_2021["ercot_difference_mw"].max(),
)

print(
    "Non-hourly gaps inside affected period:",
    (
        bad_2021["HourEnding_utc"]
        .diff()
        .dropna()
        .ne(pd.Timedelta(hours=1))
        .sum()
    ),
)

Number of affected rows: 48
First affected time: 05/17/2021 01:00
Last affected time: 05/18/2021 24:00
Minimum difference (MW): 681.7882361000011
Maximum difference (MW): 2359.831268999995
Non-hourly gaps inside affected period: 0


In [74]:
# Display the beginning and end of the affected interval.
display(
    bad_2021[
        [
            "HourEnding_raw",
            "ERCOT",
            "calculated_ercot_mw",
            "ercot_difference_mw",
        ]
    ].head()
)

display(
    bad_2021[
        [
            "HourEnding_raw",
            "ERCOT",
            "calculated_ercot_mw",
            "ercot_difference_mw",
        ]
    ].tail()
)

,HourEnding_raw,ERCOT,calculated_ercot_mw,ercot_difference_mw
3263,05/17/2021 01:00,36961.979842,36108.827384,853.152458
3264,05/17/2021 02:00,35467.187055,34699.478829,767.708226
3265,05/17/2021 03:00,34603.949944,33879.583141,724.366803
3266,05/17/2021 04:00,34203.922200,33490.479179,713.443021
3267,05/17/2021 05:00,34594.242757,33827.722440,766.520317


,HourEnding_raw,ERCOT,calculated_ercot_mw,ercot_difference_mw
3306,05/18/2021 20:00,45175.253915,43706.843311,1468.410604
3307,05/18/2021 21:00,45068.679907,43643.973905,1424.706002
3308,05/18/2021 22:00,44023.319507,42695.647160,1327.672347
3309,05/18/2021 23:00,40884.125112,39807.259975,1076.865137
3310,05/18/2021 24:00,37667.169341,36892.050366,775.118975


In [75]:
# Count zone-total mismatches by source year.
mismatch_counts = pd.Series({
    year: int(
        df["zone_total_mismatch_flag"].sum()
    )
    for year, df in cleaned_by_year.items()
}, name="zone_total_mismatch_rows")

mismatch_counts

2019     0
2020     0
2021    48
2022     0
2023     0
2024     0
2025     0
Name: zone_total_mismatch_rows, dtype: int64

## 7. Combine Cleaned Annual Datasets

Concatenate available cleaned annual frames, sort by the constructed UTC Hour Ending, and verify row counts and time continuity across year boundaries.

In [76]:
# Combine the cleaned annual datasets in chronological UTC order.
available_years = sorted(cleaned_by_year.keys())

print("Years to combine:", available_years)

annual_dataframes = [
    cleaned_by_year[year]
    for year in available_years
]

combined_load = pd.concat(
    annual_dataframes,
    ignore_index=True,
)

combined_load = combined_load.sort_values(
    "HourEnding_utc"
)

combined_load = combined_load.reset_index(
    drop=True
)

combined_load["global_interval_id"] = range(
    1,
    len(combined_load) + 1,
)

print("Combined shape:", combined_load.shape)

display(combined_load.head(3))

display(combined_load.tail(3))

Years to combine: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
Combined shape: (61368, 20)


,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT,HourEnding_raw,is_dst_label,HourEnding_local,interval_id,source_year,HourEnding_utc,calculated_ercot_mw,ercot_difference_mw,zone_total_mismatch_flag,global_interval_id
0,01/01/2019 01:00,9783.594839,1264.194059,3164.720629,827.759428,11697.766093,3066.891146,5993.974289,1282.542956,37081.443439,01/01/2019 01:00,False,2019-01-01 01:00:00,1,2019,2019-01-01 07:00:00+00:00,37081.443439,7.275958e-12,False,1
1,01/01/2019 02:00,9726.062326,1270.494294,3178.950849,830.256302,11787.936174,3140.383209,6029.773970,1295.132811,37258.989933,01/01/2019 02:00,False,2019-01-01 02:00:00,2,2019,2019-01-01 08:00:00+00:00,37258.989935,-2.000008e-06,False,2
2,01/01/2019 03:00,9654.373521,1269.504203,3200.881160,831.505553,11861.628382,3137.721342,6043.348512,1301.225133,37300.187809,01/01/2019 03:00,False,2019-01-01 03:00:00,3,2019,2019-01-01 09:00:00+00:00,37300.187806,3.000001e-06,False,3


,HourEnding,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT,HourEnding_raw,is_dst_label,HourEnding_local,interval_id,source_year,HourEnding_utc,calculated_ercot_mw,ercot_difference_mw,zone_total_mismatch_flag,global_interval_id
61365,12/31/2025 22:00,12004.088675,1710.928961,7810.496116,1659.990999,12678.814342,3648.419252,7381.238920,1363.000085,48256.977350,12/31/2025 22:00,False,2025-12-31 22:00:00,8758,2025,2026-01-01 04:00:00+00:00,48256.977350,7.275958e-12,False,61366
61366,12/31/2025 23:00,11792.978893,1678.576949,7734.808047,1594.513759,12359.687434,3619.904628,7258.624248,1355.590206,47394.684164,12/31/2025 23:00,False,2025-12-31 23:00:00,8759,2025,2026-01-01 05:00:00+00:00,47394.684164,0.000000e+00,False,61367
61367,12/31/2025 24:00,11616.065729,1647.797481,7776.223526,1623.875727,12061.995434,3589.827851,7157.546615,1337.463852,46810.796215,12/31/2025 24:00,False,2026-01-01 00:00:00,8760,2025,2026-01-01 06:00:00+00:00,46810.796215,0.000000e+00,False,61368


In [77]:
# Check the record count for each source year.
rows_by_year = combined_load.groupby(
    "source_year"
).size(
).to_frame(
    name="row_count"
)

rows_by_year

,row_count
source_year,
2019,8760
2020,8784
2021,8760
2022,8760
2023,8760
2024,8784
2025,8760


In [78]:
# Validate the combined UTC timeline and quality flags.
combined_utc_differences = combined_load[
    "HourEnding_utc"
].diff()

combined_non_hourly_intervals = combined_load.loc[
    combined_utc_differences.notna()
    & combined_utc_differences.ne(pd.Timedelta(hours=1))
].copy()

combined_row_count_matches = (
    len(combined_load) == 61368
)

combined_utc_is_unique = combined_load[
    "HourEnding_utc"
].is_unique

combined_utc_is_sorted = combined_load[
    "HourEnding_utc"
].is_monotonic_increasing

combined_non_hourly_count = len(
    combined_non_hourly_intervals
)

combined_validation = pd.Series({
    "row_count": len(combined_load),
    "expected_row_count": 61368,
    "row_count_matches": combined_row_count_matches,
    "unique_utc_timestamps": combined_utc_is_unique,
    "utc_is_sorted": combined_utc_is_sorted,
    "non_hourly_utc_intervals": combined_non_hourly_count,
    "zone_total_mismatch_rows": int(
        combined_load["zone_total_mismatch_flag"].sum()
    ),
}, name="value")

combined_validation

row_count                   61368
expected_row_count          61368
row_count_matches            True
unique_utc_timestamps        True
utc_is_sorted                True
non_hourly_utc_intervals        0
zone_total_mismatch_rows       48
Name: value, dtype: object

## 8. Save and Verify the Processed Dataset

Write the combined 2019–2025 load dataset and annual/cross-year quality summaries, then read the Parquet file back to check its structure, UTC values, and anomaly flags.

In [ ]:
# Define paths for the processed dataset and quality reports.
processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)

processed_data_path = (
    processed_dir
    / "ercot_hourly_load_2019_2025.parquet"
)

annual_validation_path = (
    processed_dir
    / "ercot_annual_validation_2019_2025.csv"
)

combined_validation_path = (
    processed_dir
    / "ercot_combined_validation_2019_2025.csv"
)

print("Processed data path:", processed_data_path.relative_to(project_root).as_posix())

print("Annual validation path:", annual_validation_path.relative_to(project_root).as_posix())

print("Combined validation path:", combined_validation_path.relative_to(project_root).as_posix())

In [82]:
# Normalize the original HourEnding field to a consistent Pandas string type.
combined_load["HourEnding"] = combined_load[
    "HourEnding"
].astype("string")

print(
    "HourEnding dtype:",
    combined_load["HourEnding"].dtype,
)

print(
    "Python value types:"
)

print(
    combined_load["HourEnding"]
    .map(lambda value: type(value).__name__)
    .value_counts()
)

HourEnding dtype: string
Python value types:
HourEnding
str    61368
Name: count, dtype: int64


In [ ]:
# Save the combined dataset as Parquet using PyArrow.
combined_load.to_parquet(
    processed_data_path,
    index=False,
    engine="pyarrow",
)

print(
    f"Processed dataset saved to: {processed_data_path.relative_to(project_root).as_posix()}"
)

**Environment note:** Install project dependencies before running the notebook with `pip install -r requirements.txt`. Do not install packages mid-analysis.

In [ ]:
# Export annual and combined quality checks to CSV.
annual_validation_output = validation_summary.rename_axis(
    "year"
).reset_index()

annual_validation_output.to_csv(
    annual_validation_path,
    index=False,
)

combined_validation_output = combined_validation.rename_axis(
    "metric"
).reset_index()

combined_validation_output.to_csv(
    combined_validation_path,
    index=False,
)

print(
    f"Annual validation saved to: {annual_validation_path.relative_to(project_root).as_posix()}"
)

print(
    f"Combined validation saved to: {combined_validation_path.relative_to(project_root).as_posix()}"
)

In [85]:
# Reload Parquet and verify the persisted dataset.
reloaded_load = pd.read_parquet(
    processed_data_path
)

shape_matches = (
    reloaded_load.shape == combined_load.shape
)

utc_values_match = reloaded_load[
    "HourEnding_utc"
].equals(
    combined_load["HourEnding_utc"]
)

reloaded_mismatch_count = int(
    reloaded_load["zone_total_mismatch_flag"].sum()
)

saved_file_validation = pd.Series({
    "reloaded_row_count": len(reloaded_load),
    "reloaded_column_count": len(reloaded_load.columns),
    "shape_matches": shape_matches,
    "utc_values_match": utc_values_match,
    "zone_total_mismatch_rows": reloaded_mismatch_count,
}, name="value")

saved_file_validation

reloaded_row_count          61368
reloaded_column_count          20
shape_matches                True
utc_values_match             True
zone_total_mismatch_rows       48
Name: value, dtype: object

---

**End of data inspection and cleaning.** The resulting dataset and validation reports provide inputs for the subsequent exploratory and weather-alignment stages.